## prepare project

In [ ]:
import sys
print(sys.version)

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
packages_name = "indextts-v2"

git_url = "https://github.com/CodingManGit/index-tts-2.git"
drive_base = "/content/drive/MyDrive/Colab Notebooks"

project_path = f"{drive_base}/{packages_name}"
repo_path = f"{project_path}/repo"

%cd "{repo_path}"

/content/drive/MyDrive/Colab Notebooks/indextts-v2/repo


In [ ]:
import os

# Ask the user for the custom branch name
branch_name = "custom-branch"

if not os.path.exists(repo_path):
  print(f"Creating directory for repository at {repo_path}")
  # Ensure the parent directory exists before cloning
  os.makedirs(os.path.dirname(repo_path), exist_ok=True)
  !git clone -b "{branch_name}" "{git_url}" "{repo_path}"
  print(f"✅ {packages_name} repository cloned to {repo_path} from branch {branch_name}.")
elif not os.path.isdir(f"{repo_path}/.git"):
  print(f"Path {repo_path} exists but is not a git repository. Please resolve this manually, likely delete it first.")
else:
  print(f"Repository already exists at {repo_path}. Skipping clone.")

Repository already exists at /content/drive/MyDrive/Colab Notebooks/indextts-v2/repo. Skipping clone.


In [ ]:
!pip uninstall -y tensorflow
# !pip uninstall -y numpy
!pip install uv
!uv pip install --system -e .

Using Python 3.12.12 environment at: /usr
Resolved 134 packages in 468ms
Prepared 2 packages in 411ms
Uninstalled 2 packages in 9ms
Installed 2 packages in 3ms
 ~ indextts==2.0.0 (from file:///content/drive/MyDrive/Colab%20Notebooks/indextts-v2/repo)
 - torchvision==0.24.0+cu126
 + torchvision==0.23.0+cu128


In [ ]:
import os
import shutil

# Asssuming below content directories are provided seperately
content_directories = ["examples", "checkpoints"]


for directory in content_directories:
    repo_dir_path = os.path.join(repo_path, directory)
    package_dir_path = os.path.join(project_path, directory)

    # 1. Ensure the target persistent directory in project_path exists
    if not os.path.exists(package_dir_path):
        if directory == "checkpoints":
          !huggingface-cli download IndexTeam/IndexTTS-2 --local-dir checkpoints
          !mv checkpoints ..
        else:
          raise ValueError

    # 2. Manage the symlink in repo_path
    if os.path.exists(repo_dir_path):
        # It exists as a regular directory or file (likely from cloning). Remove it.
        print(f"'{repo_dir_path}' exists as a non-symlink (directory/file).may need to remove it.")
        if os.path.islink(repo_dir_path):
            current_target = os.path.realpath(repo_dir_path)
            if current_target == package_dir_path:
                print(f"Symbolic link '{repo_dir_path}' already points to '{package_dir_path}'. Skipping.")
                continue
            else:
                print(f"'{repo_dir_path}' exists as a symbolic link to '{current_target}', but should point to '{package_dir_path}'. Unlinking it.")
                os.unlink(repo_dir_path)
        elif os.path.isdir(repo_dir_path):
            shutil.rmtree(repo_dir_path) # Remove directory and its contents
        else: # It's a file
            os.remove(repo_dir_path)
        os.symlink(package_dir_path, repo_dir_path)
        print(f"Created symbolic link: '{repo_dir_path}' -> '{package_dir_path}'.")
    else:
        # It does not exist at all in repo_path, so create the symlink
        os.symlink(package_dir_path, repo_dir_path)
        print(f"Created symbolic link: '{repo_dir_path}' -> '{package_dir_path}'.")

'/content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/examples' exists as a non-symlink (directory/file).may need to remove it.
Symbolic link '/content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/examples' already points to '/content/drive/MyDrive/Colab Notebooks/indextts-v2/examples'. Skipping.
'/content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/checkpoints' exists as a non-symlink (directory/file).may need to remove it.
Symbolic link '/content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/checkpoints' already points to '/content/drive/MyDrive/Colab Notebooks/indextts-v2/checkpoints'. Skipping.


In [ ]:
cache_base = "/root/.cache"
cache_runtime_tar = f"{cache_base}/cache.tar"
cache_store_tar = f"{project_path}/cache.tar"

if not os.path.exists(cache_store_tar):
  has_cache = False
else:
  has_cache = True
  !tar -xvf "{cache_store_tar}" -C "{cache_base}"

huggingface/hub/
huggingface/hub/models--nvidia--bigvgan_v2_22khz_80band_256x/
huggingface/hub/models--nvidia--bigvgan_v2_22khz_80band_256x/refs/
huggingface/hub/models--nvidia--bigvgan_v2_22khz_80band_256x/refs/main
huggingface/hub/models--nvidia--bigvgan_v2_22khz_80band_256x/snapshots/
huggingface/hub/models--nvidia--bigvgan_v2_22khz_80band_256x/snapshots/633ff708ed5b74903e86ff1298cf4a98e921c513/
huggingface/hub/models--nvidia--bigvgan_v2_22khz_80band_256x/snapshots/633ff708ed5b74903e86ff1298cf4a98e921c513/config.json
huggingface/hub/models--nvidia--bigvgan_v2_22khz_80band_256x/snapshots/633ff708ed5b74903e86ff1298cf4a98e921c513/bigvgan_generator.pt
huggingface/hub/models--nvidia--bigvgan_v2_22khz_80band_256x/blobs/
huggingface/hub/models--nvidia--bigvgan_v2_22khz_80band_256x/blobs/e95ba25972d3de0628d99cd156e9315a9c018899bf739988959ebe3544080ced
huggingface/hub/models--nvidia--bigvgan_v2_22khz_80band_256x/blobs/635bd8975629bd6d4b51c409986944a281cfe7be
huggingface/hub/models--funasr--c

## Configuration

Instead of command-line arguments, we define a configuration class to hold the settings.

In [ ]:
import os
import sys
import warnings
import html
import json
import threading
import time
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Adjust path to run from the playground directory
# try:
#     # Assumes the notebook is in the 'playground' directory, and the project root is one level up.
#     project_root = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
# except NameError:
#     # In a Jupyter notebook, __file__ is not defined.
#     project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
project_root = repo_path
sys.path.append(project_root)
# The original script adds this path, so we'll keep it for compatibility.
sys.path.append(os.path.join(project_root, "indextts"))



In [ ]:
class Config:
    verbose = False
    port = 7860
    host = "0.0.0.0"
    model_dir = os.path.join(project_root, "checkpoints")
    fp16 = False
    deepspeed = False
    cuda_kernel = False
    gui_seg_tokens = 120
    # Add gradio launch options
    # In a Colab environment, share=True is needed to get a public URL
    # inbrowser=False makes sense in a remote notebook environment
    share = True
    inbrowser = False
    debug = True

cmd_args = Config()

## Model and File Checks

In [ ]:
if not os.path.exists(cmd_args.model_dir):
    print(f"Model directory {cmd_args.model_dir} does not exist. Please download the model first.")
    # In a notebook, we'll just print the error, not exit.
    # sys.exit(1)

for file in [
    "bpe.model",
    "gpt.pth",
    "config.yaml",
    "s2mel.pth",
    "wav2vec2bert_stats.pt"
]:
    file_path = os.path.join(cmd_args.model_dir, file)
    if not os.path.exists(file_path):
        print(f"Required file {file_path} does not exist. Please download it.")
        # sys.exit(1)

## Initialize Model and Application

In [ ]:
import gradio as gr
from indextts.infer_v2 import IndexTTS2
from tools.i18n.i18n import I18nAuto

i18n = I18nAuto(language="Auto")
MODE = 'local'
tts = IndexTTS2(model_dir=cmd_args.model_dir,
                cfg_path=os.path.join(cmd_args.model_dir, "config.yaml"),
                use_fp16=cmd_args.fp16,
                use_deepspeed=cmd_args.deepspeed,
                use_cuda_kernel=cmd_args.cuda_kernel,
                )
# Supported languages list
LANGUAGES = {
    "中文": "zh_CN",
    "English": "en_US"
}
EMO_CHOICES_ALL = [i18n("与音色参考音频相同"),
                i18n("使用情感参考音频"),
                i18n("使用情感向量控制"),
                i18n("使用情感描述文本控制")]
EMO_CHOICES_OFFICIAL = EMO_CHOICES_ALL[:-1]  # skip experimental features

/usr/local/lib/python3.12/dist-packages/modelscope/hub/constants.py:46: SyntaxWarning: invalid escape sequence '\ '
  ,--.   ,--.).-'),-----. \     .'_ (,------.,--.     (_)---\_)   .-----.  .-'),-----.  _.`     \(,------.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


>> GPT weights restored from: /content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/checkpoints/gpt.pth
>> semantic_codec weights restored from: /root/.cache/huggingface/hub/models--amphion--MaskGCT/snapshots/265c6cef07625665d0c28d2faafb1415562379dc/semantic_codec/model.safetensors


/usr/local/lib/python3.12/dist-packages/audiotools/core/audio_signal.py:44: SyntaxWarning: invalid escape sequence '\_'
  Type of window to use, by default ``sqrt\_hann``.
/usr/local/lib/python3.12/dist-packages/audiotools/core/audio_signal.py:1014: SyntaxWarning: invalid escape sequence '\_'
  using functools.lru\_cache.
/usr/local/lib/python3.12/dist-packages/audiotools/core/audio_signal.py:1092: SyntaxWarning: invalid escape sequence '\_'
  """Compute how the STFT should be padded, based on match\_stride.
/usr/local/lib/python3.12/dist-packages/audiotools/core/audio_signal.py:1141: SyntaxWarning: invalid escape sequence '\_'
  Type of window to use, by default ``sqrt\_hann``.
/usr/local/lib/python3.12/dist-packages/audiotools/core/audio_signal.py:1222: SyntaxWarning: invalid escape sequence '\_'
  """Computes inverse STFT and sets it to audio\_data.


cfm loaded
length_regulator loaded
gpt_layer loaded
>> s2mel weights restored from: /content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/checkpoints/s2mel.pth
>> campplus_model weights restored from: /root/.cache/huggingface/hub/models--funasr--campplus/snapshots/fb71fe990cbf6031ae6987a2d76fe64f94377b7e/campplus_cn_common.bin
Loading weights from nvidia/bigvgan_v2_22khz_80band_256x
Removing weight norm...
>> bigvgan weights restored from: nvidia/bigvgan_v2_22khz_80band_256x


2025-12-01 10:22:26,310 WETEXT INFO found existing fst: /content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/indextts/utils/tagger_cache/zh_tn_tagger.fst
INFO:wetext-zh_normalizer:found existing fst: /content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/indextts/utils/tagger_cache/zh_tn_tagger.fst
2025-12-01 10:22:26,311 WETEXT INFO                     /content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/indextts/utils/tagger_cache/zh_tn_verbalizer.fst
INFO:wetext-zh_normalizer:                    /content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/indextts/utils/tagger_cache/zh_tn_verbalizer.fst
2025-12-01 10:22:26,312 WETEXT INFO skip building fst for zh_normalizer ...
INFO:wetext-zh_normalizer:skip building fst for zh_normalizer ...
2025-12-01 10:22:27,819 WETEXT INFO found existing fst: /usr/local/lib/python3.12/dist-packages/tn/en_tn_tagger.fst
INFO:wetext-en_normalizer:found existing fst: /usr/local/lib/python3.12/dist-packages/tn/en_tn_tagger.fst
2025-12-01 10:22:27,

>> TextNormalizer loaded
>> bpe model loaded from: /content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/checkpoints/bpe.model


In [ ]:
outputs_dir = os.path.join(project_root, "outputs")
prompts_dir = os.path.join(project_root, "prompts")
examples_dir = os.path.join(project_root, "examples")

os.makedirs(os.path.join(outputs_dir, "tasks"), exist_ok=True)
os.makedirs(prompts_dir, exist_ok=True)

MAX_LENGTH_TO_USE_SPEED = 70
example_cases = []
example_file_path = os.path.join(examples_dir, "cases.jsonl")
with open(example_file_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        example = json.loads(line)
        if example.get("emo_audio", None):
            emo_audio_path = os.path.join(examples_dir, example["emo_audio"])
        else:
            emo_audio_path = None

        example_cases.append([os.path.join(examples_dir, example.get("prompt_audio", "sample_prompt.wav")),
                              EMO_CHOICES_ALL[example.get("emo_mode", 0)],
                              example.get("text"),
                             emo_audio_path,
                             example.get("emo_weight", 1.0),
                             example.get("emo_text", ""),
                             example.get("emo_vec_1", 0),
                             example.get("emo_vec_2", 0),
                             example.get("emo_vec_3", 0),
                             example.get("emo_vec_4", 0),
                             example.get("emo_vec_5", 0),
                             example.get("emo_vec_6", 0),
                             example.get("emo_vec_7", 0),
                             example.get("emo_vec_8", 0),
                             ])

## Helper Functions

In [ ]:
def get_example_cases(include_experimental=False):
    if include_experimental:
        return example_cases  # show every example

    # exclude emotion control mode 3 (emotion from text description)
    return [x for x in example_cases if x[1] != EMO_CHOICES_ALL[3]]

def gen_single(emo_control_method, prompt, text,
               emo_ref_path, emo_weight,
               vec1, vec2, vec3, vec4, vec5, vec6, vec7, vec8,
               emo_text, emo_random,
               max_text_tokens_per_segment=120,
               *args, progress=gr.Progress()):
    output_path = None
    if not output_path:
        output_path = os.path.join(outputs_dir, f"spk_{int(time.time())}.wav")
    # set gradio progress
    tts.gr_progress = progress
    do_sample, top_p, top_k, temperature, \
        length_penalty, num_beams, repetition_penalty, max_mel_tokens = args
    kwargs = {
        "do_sample": bool(do_sample),
        "top_p": float(top_p),
        "top_k": int(top_k) if int(top_k) > 0 else None,
        "temperature": float(temperature),
        "length_penalty": float(length_penalty),
        "num_beams": num_beams,
        "repetition_penalty": float(repetition_penalty),
        "max_mel_tokens": int(max_mel_tokens),
    }
    if type(emo_control_method) is not int:
        emo_control_method = emo_control_method.value
    if emo_control_method == 0:  # emotion from speaker
        emo_ref_path = None  # remove external reference audio
    if emo_control_method == 1:  # emotion from reference audio
        pass
    if emo_control_method == 2:  # emotion from custom vectors
        vec = [vec1, vec2, vec3, vec4, vec5, vec6, vec7, vec8]
        vec = tts.normalize_emo_vec(vec, apply_bias=True)
    else:
        # don't use the emotion vector inputs for the other modes
        vec = None

    if emo_text == "":
        # erase empty emotion descriptions; `infer()` will then automatically use the main prompt
        emo_text = None

    print(f"Emo control mode:{emo_control_method},weight:{emo_weight},vec:{vec}")
    output = tts.infer(spk_audio_prompt=prompt, text=text,
                       output_path=output_path,
                       emo_audio_prompt=emo_ref_path, emo_alpha=emo_weight,
                       emo_vector=vec,
                       use_emo_text=(emo_control_method == 3), emo_text=emo_text, use_random=emo_random,
                       verbose=cmd_args.verbose,
                       max_text_tokens_per_segment=int(max_text_tokens_per_segment),
                       **kwargs)
    return gr.update(value=output, visible=True)

def update_prompt_audio():
    update_button = gr.update(interactive=True)
    return update_button

def create_warning_message(warning_text):
    return gr.HTML(f'<div style="padding: 0.5em 0.8em; border-radius: 0.5em; background: #ffa87d; color: #000; font-weight: bold">{html.escape(warning_text)}</div>')

def create_experimental_warning_message():
    return create_warning_message(i18n('提示：此功能为实验版，结果尚不稳定，我们正在持续优化中。'))

## Gradio User Interface

In [ ]:
with gr.Blocks(title="IndexTTS Demo") as demo:
    mutex = threading.Lock()
    gr.HTML('''
    <h2><center>IndexTTS2: A Breakthrough in Emotionally Expressive and Duration-Controlled Auto-Regressive Zero-Shot Text-to-Speech</h2>
<p align="center">
<a href='https://arxiv.org/abs/2506.21619'><img src='https://img.shields.io/badge/ArXiv-2506.21619-red'></a>
</p>
    ''')

    with gr.Tab(i18n("音频生成")):
        with gr.Row():
            prompt_audio = gr.Audio(label=i18n("音色参考音频"), key="prompt_audio",
                                    sources=["upload", "microphone"], type="filepath")
            prompt_list = os.listdir(prompts_dir)
            default = ''
            if prompt_list:
                default = prompt_list[0]
            with gr.Column():
                input_text_single = gr.TextArea(label=i18n("文本"), key="input_text_single", placeholder=i18n("请输入目标文本"), info=f"{i18n('当前模型版本')}{tts.model_version or '1.0'}")
                gen_button = gr.Button(i18n("生成语音"), key="gen_button", interactive=True)
            output_audio = gr.Audio(label=i18n("生成结果"), visible=True, key="output_audio")

        experimental_checkbox = gr.Checkbox(label=i18n("显示实验功能"), value=False)

        with gr.Accordion(i18n("功能设置")):
            with gr.Row():
                emo_control_method = gr.Radio(
                    choices=EMO_CHOICES_OFFICIAL,
                    type="index",
                    value=EMO_CHOICES_OFFICIAL[0], label=i18n("情感控制方式"))
                emo_control_method_all = gr.Radio(
                    choices=EMO_CHOICES_ALL,
                    type="index",
                    value=EMO_CHOICES_ALL[0], label=i18n("情感控制方式"),
                    visible=False)  # do not render

        with gr.Group(visible=False) as emotion_reference_group:
            with gr.Row():
                emo_upload = gr.Audio(label=i18n("上传情感参考音频"), type="filepath")

        with gr.Row(visible=False) as emotion_randomize_group:
            emo_random = gr.Checkbox(label=i18n("情感随机采样"), value=False)

        with gr.Group(visible=False) as emotion_vector_group:
            with gr.Row():
                with gr.Column():
                    vec1 = gr.Slider(label=i18n("喜"), minimum=0.0, maximum=1.0, value=0.0, step=0.05)
                    vec2 = gr.Slider(label=i18n("怒"), minimum=0.0, maximum=1.0, value=0.0, step=0.05)
                    vec3 = gr.Slider(label=i18n("哀"), minimum=0.0, maximum=1.0, value=0.0, step=0.05)
                    vec4 = gr.Slider(label=i18n("惧"), minimum=0.0, maximum=1.0, value=0.0, step=0.05)
                with gr.Column():
                    vec5 = gr.Slider(label=i18n("厌恶"), minimum=0.0, maximum=1.0, value=0.0, step=0.05)
                    vec6 = gr.Slider(label=i18n("低落"), minimum=0.0, maximum=1.0, value=0.0, step=0.05)
                    vec7 = gr.Slider(label=i18n("惊喜"), minimum=0.0, maximum=1.0, value=0.0, step=0.05)
                    vec8 = gr.Slider(label=i18n("平静"), minimum=0.0, maximum=1.0, value=0.0, step=0.05)

        with gr.Group(visible=False) as emo_text_group:
            create_experimental_warning_message()
            with gr.Row():
                emo_text = gr.Textbox(label=i18n("情感描述文本"),
                                      placeholder=i18n("请输入情绪描述（或留空以自动使用目标文本作为情绪描述）"),
                                      value="",
                                      info=i18n("例如：委屈巴巴、危险在悄悄逼近"))

        with gr.Row(visible=False) as emo_weight_group:
            emo_weight = gr.Slider(label=i18n("情感权重"), minimum=0.0, maximum=1.0, value=0.65, step=0.01)

        with gr.Accordion(i18n("高级生成参数设置"), open=False, visible=True) as advanced_settings_group:
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown(f"**{i18n('GPT2 采样设置')}** _{i18n('参数会影响音频多样性和生成速度详见')} [Generation strategies](https://huggingface.co/docs/transformers/main/en/generation_strategies)._ ")
                    with gr.Row():
                        do_sample = gr.Checkbox(label="do_sample", value=True, info=i18n("是否进行采样"))
                        temperature = gr.Slider(label="temperature", minimum=0.1, maximum=2.0, value=0.8, step=0.1)
                    with gr.Row():
                        top_p = gr.Slider(label="top_p", minimum=0.0, maximum=1.0, value=0.8, step=0.01)
                        top_k = gr.Slider(label="top_k", minimum=0, maximum=100, value=30, step=1)
                        num_beams = gr.Slider(label="num_beams", value=3, minimum=1, maximum=10, step=1)
                    with gr.Row():
                        repetition_penalty = gr.Number(label="repetition_penalty", precision=None, value=10.0, minimum=0.1, maximum=20.0, step=0.1)
                        length_penalty = gr.Number(label="length_penalty", precision=None, value=0.0, minimum=-2.0, maximum=2.0, step=0.1)
                    max_mel_tokens = gr.Slider(label="max_mel_tokens", value=1500, minimum=50, maximum=tts.cfg.gpt.max_mel_tokens, step=10, info=i18n("生成Token最大数量，过小导致音频被截断"), key="max_mel_tokens")
                with gr.Column(scale=2):
                    gr.Markdown(f'**{i18n("分句设置")}** _{i18n("参数会影响音频质量和生成速度")} _')
                    with gr.Row():
                        initial_value = max(20, min(tts.cfg.gpt.max_text_tokens, cmd_args.gui_seg_tokens))
                        max_text_tokens_per_segment = gr.Slider(
                            label=i18n("分句最大Token数"), value=initial_value, minimum=20, maximum=tts.cfg.gpt.max_text_tokens, step=2, key="max_text_tokens_per_segment",
                            info=i18n("建议80~200之间，值越大，分句越长；值越小，分句越碎；过小过大都可能导致音频质量不高"),
                        )
                    with gr.Accordion(i18n("预览分句结果"), open=True) as segments_settings:
                        segments_preview = gr.Dataframe(
                            headers=[i18n("序号"), i18n("分句内容"), i18n("Token数")],
                            key="segments_preview",
                            wrap=True,
                        )
            advanced_params = [
                do_sample, top_p, top_k, temperature,
                length_penalty, num_beams, repetition_penalty, max_mel_tokens,
            ]

        example_table = gr.Dataset(label="Examples",
            samples_per_page=20,
            samples=get_example_cases(include_experimental=False),
            type="values",
            components=[prompt_audio,
                        emo_control_method_all,  # important: support all mode labels!
                        input_text_single,
                        emo_upload,
                        emo_weight,
                        emo_text,
                        vec1, vec2, vec3, vec4, vec5, vec6, vec7, vec8]
        )

    def on_example_click(example):
        print(f"Example clicked: ({len(example)} values) = {example!r}")
        return (
            gr.update(value=example[0]),
            gr.update(value=example[1]),
            gr.update(value=example[2]),
            gr.update(value=example[3]),
            gr.update(value=example[4]),
            gr.update(value=example[5]),
            gr.update(value=example[6]),
            gr.update(value=example[7]),
            gr.update(value=example[8]),
            gr.update(value=example[9]),
            gr.update(value=example[10]),
            gr.update(value=example[11]),
            gr.update(value=example[12]),
            gr.update(value=example[13]),
        )

    example_table.click(on_example_click,
                        inputs=[example_table],
                        outputs=[prompt_audio,
                                 emo_control_method,
                                 input_text_single,
                                 emo_upload,
                                 emo_weight,
                                 emo_text,
                                 vec1, vec2, vec3, vec4, vec5, vec6, vec7, vec8]
    )

    def on_input_text_change(text, max_text_tokens_per_segment):
        if text and len(text) > 0:
            text_tokens_list = tts.tokenizer.tokenize(text)

            segments = tts.tokenizer.split_segments(text_tokens_list, max_text_tokens_per_segment=int(max_text_tokens_per_segment))
            data = []
            for i, s in enumerate(segments):
                segment_str = ''.join(s)
                tokens_count = len(s)
                data.append([i, segment_str, tokens_count])
            return {
                segments_preview: gr.update(value=data, visible=True, type="array"),
            }
        else:
            df = pd.DataFrame([], columns=[i18n("序号"), i18n("分句内容"), i18n("Token数")])
            return {
                segments_preview: gr.update(value=df),
            }

    def on_method_change(emo_control_method):
        if emo_control_method == 1:  # emotion reference audio
            return (
                gr.update(visible=True),
                gr.update(visible=False),
                gr.update(visible=False),
                gr.update(visible=False),
                gr.update(visible=True)
            )
        elif emo_control_method == 2:  # emotion vectors
            return (
                gr.update(visible=False),
                gr.update(visible=True),
                gr.update(visible=True),
                gr.update(visible=False),
                gr.update(visible=True)
            )
        elif emo_control_method == 3:  # emotion text description
            return (
                gr.update(visible=False),
                gr.update(visible=True),
                gr.update(visible=False),
                gr.update(visible=True),
                gr.update(visible=True)
            )
        else:  # 0: same as speaker voice
            return (
                gr.update(visible=False),
                gr.update(visible=False),
                gr.update(visible=False),
                gr.update(visible=False),
                gr.update(visible=False)
            )

    emo_control_method.change(on_method_change,
        inputs=[emo_control_method],
        outputs=[emotion_reference_group,
                 emotion_randomize_group,
                 emotion_vector_group,
                 emo_text_group,
                 emo_weight_group]
    )

    def on_experimental_change(is_experimental, current_mode_index):
        new_choices = EMO_CHOICES_ALL if is_experimental else EMO_CHOICES_OFFICIAL
        new_index = current_mode_index if current_mode_index < len(new_choices) else 0

        return (
            gr.update(choices=new_choices, value=new_choices[new_index]),
            gr.update(samples=get_example_cases(include_experimental=is_experimental)),
        )

    experimental_checkbox.change(
        on_experimental_change,
        inputs=[experimental_checkbox, emo_control_method],
        outputs=[emo_control_method, example_table]
    )

    input_text_single.change(
        on_input_text_change,
        inputs=[input_text_single, max_text_tokens_per_segment],
        outputs=[segments_preview]
    )

    max_text_tokens_per_segment.change(
        on_input_text_change,
        inputs=[input_text_single, max_text_tokens_per_segment],
        outputs=[segments_preview]
    )

    prompt_audio.upload(update_prompt_audio,
                         inputs=[],
                         outputs=[gen_button])

    gen_button.click(gen_single,
                     inputs=[
                         emo_control_method, prompt_audio, input_text_single, emo_upload, emo_weight,
                         vec1, vec2, vec3, vec4, vec5, vec6, vec7, vec8,
                         emo_text, emo_random,
                         max_text_tokens_per_segment,
                         *advanced_params,
                     ],
                     outputs=[output_audio])

## Launch the Application

Run the following cell to start the Gradio web server.

In [ ]:
demo.queue(20)
demo.launch(
    server_name=cmd_args.host,
    server_port=cmd_args.port,
    share=cmd_args.share,
    inbrowser=cmd_args.inbrowser,
    debug=cmd_args.debug,
    allowed_paths=[os.path.join(os.getcwd(), 'examples')]
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b0e848c986099e5df1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Example clicked: (14 values) = ['/content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/examples/voice_01.wav', 'Same as the voice reference', 'Translate for me, what is a surprise!', None, 1.0, '', 0, 0, 0, 0, 0, 0, 0, 0]
Emo control mode:0,weight:1,vec:None
>> starting inference...
Use the specified emotion vector


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.53.0. You should pass an instance of `Cache` instead, e.g. `past_key_values=DynamicCache.from_legacy_cache(past_key_values)`.
100%|██████████| 25/25 [00:00<00:00, 32.32it/s]


torch.Size([1, 74240])
>> gpt_gen_time: 4.38 seconds
>> gpt_forward_time: 0.02 seconds
>> s2mel_time: 0.80 seconds
>> bigvgan_time: 0.31 seconds
>> Total inference time: 10.29 seconds
>> Generated audio length: 3.37 seconds
>> RTF: 3.0548
>> wav file saved to: /content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/outputs/spk_1764584596.wav
Emo control mode:0,weight:1,vec:None
>> starting inference...
Use the specified emotion vector


100%|██████████| 25/25 [00:01<00:00, 13.93it/s]


...

torch.Size([1, 242176])
Use the specified emotion vector


100%|██████████| 25/25 [00:01<00:00, 20.46it/s]


torch.Size([1, 68096])
>> gpt_gen_time: 28.66 seconds
>> gpt_forward_time: 0.06 seconds
>> s2mel_time: 5.58 seconds
>> bigvgan_time: 0.87 seconds
>> Total inference time: 36.07 seconds
>> Generated audio length: 24.63 seconds
>> RTF: 1.4644
>> wav file saved to: /content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/outputs/spk_1764592272.wav
Emo control mode:0,weight:0.5,vec:None
>> starting inference...
Use the specified emotion vector


100%|██████████| 25/25 [00:01<00:00, 17.29it/s]


torch.Size([1, 132864])
>> gpt_gen_time: 6.90 seconds
>> gpt_forward_time: 0.02 seconds
>> s2mel_time: 1.48 seconds
>> bigvgan_time: 0.19 seconds
>> Total inference time: 9.05 seconds
>> Generated audio length: 6.03 seconds
>> RTF: 1.5023
>> wav file saved to: /content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/outputs/spk_1764592842.wav
Example clicked: (14 values) = ['/content/drive/MyDrive/Colab Notebooks/indextts-v2/repo/examples/voice_09.wav', 'Use emotion vectors', '对不起嘛！我的记性真的不太好，但是和你在一起的事情，我都会努力记住的~', None, 0.8, '', 0, 0, 0.8, 0, 0, 0, 0, 0]


In [ ]:
if not has_cache:
  !tar -cvf "{cache_runtime_tar}" -C "{cache_base}" "torch_extensions" "huggingface/hub"
  !cp "{cache_runtime_tar}" "{cache_store_tar}"

In [ ]:
if not os.path.exists(f"{repo_path}/setup_env.ipynb"):
  !mv "{project_path}/setup_env.ipynb" "{repo_path}/setup_env.ipynb"

In [ ]:
!uv pip list | grep protobuf

In [ ]:
!uv pip show protobuf